In [12]:
import time
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [13]:
df = pd.read_csv("ToyotaCorolla.csv")
df.head()

,Id,Model,Price,Age_08_04,Mfg_Month,Mfg_Year,KM,Fuel_Type,HP,Met_Color,...,Powered_Windows,Power_Steering,Radio,Mistlamps,Sport_Model,Backseat_Divider,Metallic_Rim,Radio_cassette,Parking_Assistant,Tow_Bar
0,1,TOYOTA Corolla 2.0 D4D HATCHB TERRA 2/3-Doors,13500,23,10,2002,46986,Diesel,90,1,...,1,1,0,0,0,1,0,0,0,0
1,2,TOYOTA Corolla 2.0 D4D HATCHB TERRA 2/3-Doors,13750,23,10,2002,72937,Diesel,90,1,...,0,1,0,0,0,1,0,0,0,0
2,3,TOYOTA Corolla 2.0 D4D HATCHB TERRA 2/3-Doors,13950,24,9,2002,41711,Diesel,90,1,...,0,1,0,0,0,1,0,0,0,0
3,4,TOYOTA Corolla 2.0 D4D HATCHB TERRA 2/3-Doors,14950,26,7,2002,48000,Diesel,90,0,...,0,1,0,0,0,1,0,0,0,0
4,5,TOYOTA Corolla 2.0 D4D HATCHB SOL 2/3-Doors,13750,30,3,2002,38500,Diesel,90,0,...,1,1,0,1,0,1,0,0,0,0


In [14]:
df = df.drop(columns=["Id"], errors="ignore")


y = df["Price"]
X = df.drop(columns=["Price"])


X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median())

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

##### Сделали ли вы предобработку данных для случайного леса? Отличалась ли она от предобработки данных для линейной модели? Если до, то почему?
- заполнил пропуски и применил OHE. В отличие от линйеной модели не применял масштабирование, так как деревья не чуствительны к масштабн
    
##### Как именно вы разделили выборку?
- test 80%, train 20%

##### На сколько частей нужно делить выборку при использовании кросс-валидации?
- Часто делят на 5 частей

##### Можно ли не использовать кросс-валидацию? Если да, то как делить выборку в таком случае
- Да, делить тогда нужно например на test 80%, train 20%

In [16]:
tree = DecisionTreeRegressor(max_depth=5, random_state=42)
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

start = time.time()
tree.fit(X_train, y_train)
print("Tree time:", time.time() - start)

start = time.time()
rf.fit(X_train, y_train)
print("RF time:", time.time() - start)

Tree time: 0.02211165428161621
RF time: 0.5920403003692627


##### Сравнение скорости: сравните, какая модель обучалась быстрее.
- Одно дерево обучается быстрее

##### Можно ли добиться одинаковой или близкой к одинаковой скорости?
- Да, если уменьшить n_estimators

In [17]:
def evaluate(model, X_train, X_test, y_train, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))
    print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))
    print("MAE:", mean_absolute_error(y_test, y_pred_test))
    print("R2:", r2_score(y_test, y_pred_test))
    print("------")

In [18]:
print("Decision Tree")
evaluate(tree, X_train, X_test, y_train, y_test)

print("Random Forest")
evaluate(rf, X_train, X_test, y_train, y_test)

Decision Tree
Train RMSE: 1020.8944570796871
Test RMSE: 1213.1724318441595
MAE: 903.7837755423543
R2: 0.8896941736373649
------
Random Forest
Train RMSE: 938.3398943129409
Test RMSE: 1044.581589259936
MAE: 792.8467219695625
R2: 0.9182216919264317
------


1. **Какие метрики вы использовали для сравнения моделей?** (Обоснуйте выбор: например, почему RMSE, а не MAE, или наоборот. Зачем нужен $R^2$?).
- Я использовал RMSE, MAE, $R^2$?. RMSE показывает ошидку в тех же единицах, что и ```y```, MAE показывает среднюю ошибку.
2. **На какой части выборки вы считали метрики?** 
- Считал на train и test для выявлению переобучения. Итоговая оценка модели считается на test выборке.
3. **Какая модель по итогу справилась лучше?**
- Лес справился лучше, так как он лучше обобщает данные и менее склонен к переобучению
4. **Насколько хорошие получились результаты?** 
- RMSE ~1000, значит модель в среднем ошибается на 1000$(что мало по сравнению с ценой автомобиля), также доволно высокое значение $R^2$